In [2]:
# !git clone -b dev https://github.com/justinshenk/temporal-manifolds.git
# %cd temporal-manifolds
# !gcloud auth application-default login
# !mv -n .env.example .env

Cloning into 'temporal-manifolds'...
remote: Enumerating objects: 553, done.
remote: Counting objects: 100% (553/553), done.
remote: Compressing objects: 100% (307/307), done.
remote: Total 553 (delta 306), reused 438 (delta 191), pack-reused 0 (from 0)
Receiving objects: 100% (553/553), 527.91 KiB | 3.52 MiB/s, done.
Resolving deltas: 100% (306/306), done.
/content/temporal-manifolds
Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=lFcBPcRww3AVQjgkLeYesAoRKgN5JL&prompt=consent&token_usage=remote&access_type=offline&code_challenge=ufFUVFZ5mSQsj

# Download and aggregate filtered activation files

Download a user-selected completions file from GCS, filter its records by template metadata, download their activation caches, and aggregate MLP, attention, and residual activations over cached token positions. GCP project and bucket settings are read from `.env`; missing Application Default Credentials trigger `gcloud auth application-default login`.

In [3]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from temporal_manifolds.activations.extract_activations import load_selected_node_groups
from temporal_manifolds.utils.activation_aggregation import (
    aggregate_activation_file,
    nodes_for_classes,
)
from temporal_manifolds.utils.completion_filters import (
    download_activation_files,
    find_activation_paths,
)

## Configuration

The completions file is downloaded from the hardcoded GCS object to `COMPLETIONS_PATH` using authenticated Google Cloud APIs. Set either template metadata filter or `TASK_METADATA_FILTERS` to `None` to disable it. Task metadata filters require every configured key-value pair to match. `NODE_CLASSES` unions the requested selected-node classes. Residual streams are included atomically as complete `layer_out/<layer>` tensors: `None` includes every layer, a set includes only those layers, and `set()` excludes all residual streams. `MAX_FILES` defaults to 10 to avoid an unexpectedly large download.

In [32]:
PROMPT_FRAMING: str | None = "task_available_time"
OUTPUT_FORMAT: str | None = "strategy_steps"
TASK_METADATA_FILTERS: dict[str, object] | None = {
    "difficulty": "low",
    "domain": "communication",
}
COMPLETIONS_PATH = repo_root / "data" / "completions_256.jsonl"
COMPLETIONS_GCS_BUCKET = "temporal-research-bucket"
COMPLETIONS_GCS_PREFIX = "completions"
ACTIVATIONS_DIR = repo_root / "results" / "feature_geometry_after_assistant_residual_stream"
SELECTED_NODES_PATH = repo_root / "data" / "selected_nodes" / "final_500_eap_ig.pkl"
NODE_CLASSES: set[str] | None = {"p_generic", "n_generic"}
RESIDUAL_STREAM_LAYERS: set[int] | None = set()
AGGREGATION_POLICY = "assistant"  # assistant or all
MAX_FILES: int | None = 1000000
OVERWRITE = False

## Download completions

Download `gs://temporal-research-bucket/completions/completions_256.jsonl` through the authenticated Google Cloud Storage API before filtering it. An existing local file is skipped unless `OVERWRITE` is `True`.

In [8]:
download_activation_files(
    [COMPLETIONS_PATH],
    gcs_prefix=COMPLETIONS_GCS_PREFIX,
    overwrite=OVERWRITE,
    upload_root=COMPLETIONS_PATH.parent,
    bucket_name=COMPLETIONS_GCS_BUCKET,
)
print(f"Completions file: {COMPLETIONS_PATH}")

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Completions file: /content/temporal-manifolds/data/completions_256.jsonl


In [11]:
matching_paths = find_activation_paths(
    prompt_framing=PROMPT_FRAMING,
    output_format=OUTPUT_FORMAT,
    task_metadata=TASK_METADATA_FILTERS,
    completions_path=COMPLETIONS_PATH,
    activations_dir=ACTIVATIONS_DIR,
)
paths_to_download = matching_paths if MAX_FILES is None else matching_paths[:MAX_FILES]

print(f"Found {len(matching_paths):,} matching activation files.")
print(f"Selected {len(paths_to_download):,} files for download.")
for path in paths_to_download[:10]:
    print(path)

Found 154 matching activation files.
Selected 154 files for download.
/content/temporal-manifolds/results/feature_geometry_after_assistant_residual_stream/activations_sample_00000.pt
/content/temporal-manifolds/results/feature_geometry_after_assistant_residual_stream/activations_sample_00001.pt
/content/temporal-manifolds/results/feature_geometry_after_assistant_residual_stream/activations_sample_00002.pt
/content/temporal-manifolds/results/feature_geometry_after_assistant_residual_stream/activations_sample_00003.pt
/content/temporal-manifolds/results/feature_geometry_after_assistant_residual_stream/activations_sample_00004.pt
/content/temporal-manifolds/results/feature_geometry_after_assistant_residual_stream/activations_sample_00005.pt
/content/temporal-manifolds/results/feature_geometry_after_assistant_residual_stream/activations_sample_00006.pt
/content/temporal-manifolds/results/feature_geometry_after_assistant_residual_stream/activations_sample_00007.pt
/content/temporal-manifold

## Download activation caches

Existing local files are skipped unless `OVERWRITE` is `True`. Activation files are downloaded from the hardcoded `conversational_after_assistant_residual_stream/results/feature_geometry_after_assistant_residual_stream` GCS path. The selected-node definitions use the repository-relative GCS convention.

In [12]:
downloaded_paths = download_activation_files(
    paths_to_download,
    gcs_prefix=(
        "conversational_after_assistant_residual_stream/"
        "results/feature_geometry_after_assistant_residual_stream"
    ),
    overwrite=OVERWRITE,
    upload_root=ACTIVATIONS_DIR,
    bucket_name="temporal-research-bucket",
)
download_activation_files(
    [SELECTED_NODES_PATH],
    gcs_prefix="eap-ig/data/selected_nodes",
    upload_root=SELECTED_NODES_PATH.parent,
    bucket_name="temporal-research-bucket",
)

print(f"Download complete: {len(downloaded_paths):,} activation paths.")
print(f"Selected-node definitions: {SELECTED_NODES_PATH}")

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Download complete: 154 activation paths.
Selected-node definitions: /content/temporal-manifolds/data/selected_nodes/final_500_eap_ig.pkl


## Aggregate cached positions

Cached tensors begin as `batch x cached positions x features`; selected attention tensors additionally retain their head-feature dimension. The same policy is applied to every included MLP, attention, and complete residual-stream tensor:

- `assistant`: keep the first cached position.
- `all`: average every cached token position.

In [33]:
if not downloaded_paths:
    raise ValueError("No activation files were selected.")

selected_node_groups = load_selected_node_groups(SELECTED_NODES_PATH)
print("Available node classes:", sorted(selected_node_groups))
allowed_nodes = nodes_for_classes(selected_node_groups, NODE_CLASSES)
print("Node-class filter:", "all" if NODE_CLASSES is None else sorted(NODE_CLASSES))
print(
    "Residual-stream layers:",
    "all" if RESIDUAL_STREAM_LAYERS is None else sorted(RESIDUAL_STREAM_LAYERS),
)

aggregated_by_file = []
for path in downloaded_paths:
    aggregated_by_file.append(
        aggregate_activation_file(
            path,
            AGGREGATION_POLICY,
            allowed_nodes,
            RESIDUAL_STREAM_LAYERS,
        )
    )

print(f"Aggregated {len(aggregated_by_file):,} files with policy={AGGREGATION_POLICY!r}.")
for activation_type, tensors in aggregated_by_file[0]["activations"].items():
    example_shapes = {name: tuple(tensor.shape) for name, tensor in list(tensors.items())[:3]}
    print(activation_type, example_shapes)
print("Retained selected nodes:", {
    name: len(indices)
    for name, indices in aggregated_by_file[0]["node_indices"].items()
})
print("Included residual streams:", aggregated_by_file[0]["included_residual_streams"])

Available node classes: ['n_common_LT', 'n_common_ST', 'n_generic', 'p_common_LT', 'p_common_ST', 'p_generic', 'sym_LT_p', 'sym_ST_p']
Node-class filter: ['n_generic', 'p_generic']
Residual-stream layers: []
Aggregated 154 files with policy='assistant'.
mlp {'mlp_hidden/20': (2,), 'mlp_hidden/21': (2,), 'mlp_hidden/22': (2,)}
attn {'z/14': (1, 128), 'z/15': (2, 128), 'z/16': (1, 128)}
residual {}
Retained selected nodes: {'mlp_hidden/20': 2, 'mlp_hidden/21': 2, 'mlp_hidden/22': 2, 'mlp_hidden/24': 1, 'mlp_hidden/25': 3, 'mlp_hidden/27': 4, 'mlp_hidden/29': 2, 'mlp_hidden/30': 3, 'mlp_hidden/32': 5, 'mlp_hidden/33': 4, 'mlp_hidden/34': 5, 'mlp_hidden/35': 2, 'z/14': 1, 'z/15': 2, 'z/16': 1, 'z/17': 1, 'z/18': 1, 'z/19': 4, 'z/20': 1, 'z/21': 5, 'z/22': 4, 'z/23': 2, 'z/24': 1, 'z/26': 3, 'z/29': 2, 'z/30': 2, 'z/32': 4, 'mlp_hidden/13': 1, 'mlp_hidden/15': 1, 'mlp_hidden/23': 1, 'mlp_hidden/28': 1, 'mlp_hidden/31': 1, 'z/28': 1, 'z/33': 1, 'z/34': 2}
Included residual streams: []


In [34]:
import torch

coll=[]
for f in aggregated_by_file:
    temp=[]
    for mlp_act in f["activations"]["mlp"].values():
        temp.append(mlp_act)
    for att_act in f["activations"]["attn"].values():
        temp.append(att_act.ravel())
    temp = torch.cat(temp, dim=0)
    coll.append(temp)

In [39]:
A = torch.stack(coll).to(torch.float)

In [40]:
U,S,V = torch.pca_lowrank(A)

In [43]:
dirs = V[:, :3]

In [44]:
projs = A @ dirs

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

# Prepare projections and their corresponding time horizons.
projs_np = projs.detach().cpu().numpy()

sample_indices = [
    int(path.stem.removeprefix("activations_sample_"))
    for path in downloaded_paths
]
target_indices = set(sample_indices)
completion_metadata = {}
with COMPLETIONS_PATH.open(encoding="utf-8") as completion_file:
    for index, line in enumerate(completion_file):
        if index in target_indices:
            completion_metadata[index] = json.loads(line)["prompt_metadata"]
            if len(completion_metadata) == len(target_indices):
                break

missing_indices = target_indices - completion_metadata.keys()
if missing_indices:
    raise ValueError(f"Missing completion metadata for sample indices: {sorted(missing_indices)}")

# The supplied conversion implies an average month of 2,628,000 seconds.
seconds_to_months = 3.80517e-7
unit_to_months = {
    "second": seconds_to_months,
    "seconds": seconds_to_months,
    "minute": 60 * seconds_to_months,
    "minutes": 60 * seconds_to_months,
    "hour": 60 * 60 * seconds_to_months,
    "hours": 60 * 60 * seconds_to_months,
    "day": 24 * 60 * 60 * seconds_to_months,
    "days": 24 * 60 * 60 * seconds_to_months,
    "week": 7 * 24 * 60 * 60 * seconds_to_months,
    "weeks": 7 * 24 * 60 * 60 * seconds_to_months,
    "month": 1.0,
    "months": 1.0,
    "year": 12.0,
    "years": 12.0,
    "decade": 120.0,
    "decades": 120.0,
    "century": 1_200.0,
    "centuries": 1_200.0,
    "millennium": 12_000.0,
    "millennia": 12_000.0,
}

time_horizon_months = []
for index in sample_indices:
    metadata = completion_metadata[index]
    # Prefer the canonical base horizon so alternate unit renderings get the same color.
    value = float(metadata.get("base_value", metadata["value"]))
    unit = metadata.get("base_unit", metadata["unit"])
    time_horizon_months.append(value * unit_to_months[unit.lower()])

time_horizon_months = np.asarray(time_horizon_months)
if np.any(time_horizon_months <= 0):
    raise ValueError("Time horizons must be positive before taking the logarithm.")
log_time_horizon_months = np.log10(time_horizon_months)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    projs_np[:, 0],
    projs_np[:, 1],
    projs_np[:, 2],
    c=log_time_horizon_months,
    cmap="viridis",
    marker="o",
    alpha=0.6,
)
fig.colorbar(scatter, ax=ax, pad=0.1, label="log10(time horizon in months)")

ax.set_xlabel('Component 1')
ax.set_ylabel('Component 2')
ax.set_zlabel('Component 3')
ax.set_title('3D Scatter Plot Colored by Time Horizon')

plt.show()

In [ ]:
import plotly.express as px
import pandas as pd

df_projs = pd.DataFrame(projs_np, columns=['PC1', 'PC2', 'PC3'])
df_projs["time_horizon_months"] = time_horizon_months
df_projs["log10_time_horizon_months"] = log_time_horizon_months

fig = px.scatter_3d(
    df_projs,
    x='PC1',
    y='PC2',
    z='PC3',
    color="log10_time_horizon_months",
    color_continuous_scale="Viridis",
    hover_data={"time_horizon_months": ":.6g"},
    labels={"log10_time_horizon_months": "log10(time horizon in months)"},
    title='Interactive 3D Scatter Plot Colored by Time Horizon',
    opacity=0.7
)

fig.update_traces(marker=dict(size=4))
fig.show()